In [1]:
!g++ --version

g++ (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



In [2]:
!nvidia-smi;
!nvcc --version;

Thu Jul  2 20:23:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%pip install meson ninja uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 22.3 MB/s eta 0:00:00


In [5]:
%%bash
cd /content/drive/MyDrive/cuda-gnn-inference;
git pull;

Already up to date.


In [ ]:
%%bash
cd /content/drive/MyDrive/cuda-gnn-inference;
uv sync --frozen

## Test semi-completo CPU/GPU

Le celle seguenti compilano il progetto, generano un piccolo grafo sintetico ed eseguono lo stesso workload con i backend sequenziale, OpenMP e CUDA. Gli output numerici vengono confrontati automaticamente con una tolleranza per i calcoli floating point.

In [ ]:
%%bash
set -euo pipefail
PROJECT=/content/drive/MyDrive/cuda-gnn-inference
cd "$PROJECT"

echo '=== GPU disponibile ==='
nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader
echo '=== Toolchain CUDA ==='
nvcc --version

echo '=== Configurazione e compilazione ==='
if [ -d builddir/meson-private ]; then
  meson setup --reconfigure builddir
else
  meson setup builddir
fi
meson compile -C builddir
./builddir/gnn --help


In [ ]:
%%bash
set -euo pipefail
PROJECT=/content/drive/MyDrive/cuda-gnn-inference
TEST_DATA=/content/cuda-gnn-semi-test
mkdir -p "$TEST_DATA"
cd "$PROJECT"

echo '=== Generazione workload sintetico ==='
uv run python scripts/synthetic_generator.py \
  --type barabasi_albert \
  --nodes 256 \
  --feature_dim 32 \
  --m 4 \
  --seed 42 \
  --out_prefix "$TEST_DATA/graph"
ls -lh "$TEST_DATA/graph.bin_graph" "$TEST_DATA/graph_feats.bin_matrix"


In [ ]:
from pathlib import Path
import re
import subprocess
import numpy as np

project = Path('/content/drive/MyDrive/cuda-gnn-inference')
executable = project / 'builddir' / 'gnn'
test_data = Path('/content/cuda-gnn-semi-test')

def run_backend(mode, arguments=()):
    command = [str(executable), mode, *map(str, arguments)]
    result = subprocess.run(command, cwd=project, text=True, capture_output=True)
    print(f'\n$ {" ".join(command)}')
    print(result.stdout, end='')
    if result.stderr:
        print(result.stderr, end='')
    if result.returncode != 0:
        raise RuntimeError(f'{mode} terminato con codice {result.returncode}')
    rows = []
    for line in result.stdout.splitlines():
        match = re.fullmatch(r'\s*\[([^]]+)\]', line)
        if match:
            rows.append([float(value) for value in match.group(1).split(',')])
    if not rows:
        raise RuntimeError(f'{mode} non ha prodotto righe numeriche')
    return np.asarray(rows, dtype=np.float32)

def compare_case(name, arguments=()):
    print(f'\n========== {name} ==========')
    sequential = run_backend('sequential', arguments)
    parallel = run_backend('parallel', arguments)
    cuda = run_backend('cuda', arguments)
    np.testing.assert_allclose(parallel, sequential, rtol=1e-5, atol=1e-5)
    np.testing.assert_allclose(cuda, sequential, rtol=1e-5, atol=1e-5)
    print(f'OK: {name} - righe stampate equivalenti, shape confrontata {cuda.shape}')

compare_case('Demo integrato')
compare_case(
    'Grafo sintetico caricato da file',
    (test_data / 'graph.bin_graph', test_data / 'graph_feats.bin_matrix'),
)
print('\nTUTTI I TEST SONO PASSATI')
